# 05. Grounding Answers With the Built-in Web Search Tool

This notebook (**AI-103 -> Section 01**) attaches the Responses API's built-in `web_search` tool to a request, letting the model ground its answer in live web results (with citations) instead of relying solely on its static training data.

**Difficulty: Intermediate**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `openai`, `azure-identity`, `python-dotenv`

**Azure resources**
- An Azure OpenAI / AI Foundry deployment with the built-in `web_search` tool available. Availability of built-in tools can depend on the model/region/deployment configuration -- check your Foundry project if you get a tool-not-supported error.
- Entra ID auth via `az login`.

**Environment variables** -- add to the root `.env`:
```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_DEPLOYMENT=gpt-4.1
```


## What You'll Learn
- Built-in, server-side Responses API tools vs. custom function tools
- The `tool_choice` parameter (`auto`, `required`, `none`, or naming a specific tool)
- Live web grounding vs. relying on a model's static training knowledge
- When to reach for web search vs. your own private-document grounding (RAG)


### 1. Imports and configuration


In [ ]:
import os

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")


### 2. Build the client


In [ ]:
client = OpenAI(
    base_url=endpoint,
    api_key=token_provider,
)


### 3. Send a request with the `web_search` tool attached

`tools=[{"type": "web_search"}]` attaches the built-in web search tool; `tool_choice="auto"` lets the model itself decide whether the question actually needs a web lookup or can be answered directly.

**Exam tip:** Distinguish two tool categories in the Responses API: **built-in/hosted tools** (`web_search`, `code_interpreter`, `file_search`) which the service executes entirely server-side and returns results for in the same response; and **custom function tools**, where the model only returns a proposed function name + arguments and *your application code* must execute it and send the result back in a follow-up call. AI-102 scenario questions often hinge on this "who actually executes the tool" distinction. `tool_choice` accepts `"auto"` (model decides), `"required"` (must call some tool), `"none"` (never call a tool), or an object naming one specific tool to force.

**Alternatives:** When the answer must come from your own private/proprietary documents rather than the public internet, ground the model in your own knowledge base instead -- via the `file_search` built-in tool, or an Azure AI Search-backed RAG agent like the one referenced in `03. Section Code/01_conversation.py`.


In [ ]:
response = client.responses.create(
    model=deployment_name,
    instructions="You are a helpful research assistant. Always cite your sources.",
    input="What are the latest developments in AI regulation in the European Union?",
    tools=[{"type": "web_search"}],
    tool_choice="auto",
)


### 4. Read the answer


In [ ]:
print(f"answer: {response.output_text}")


## Summary

You attached the built-in `web_search` tool and let the model decide (`tool_choice="auto"`) whether to use it, producing an answer grounded in current web results rather than only the model's training data.


## Try It Yourself
1. **Easy:** Change the question to a different current-events topic.
2. **Medium:** Set `tool_choice="none"` and compare the output -- expect it to fall back to the model's static training knowledge, likely with a knowledge-cutoff caveat and no citations.
3. **Advanced:** Inspect `response.output` for the web-search tool-call item (similar to the `code_interpreter_call` item inspected in the next notebook) and print the specific sources it used, rather than relying on the model mentioning them in prose.
